## This module contains a collection of functions designed for analyzing pandas DataFrames and displaying related statistics.

In [1]:
import textwrap
import pandas   as pd

In [2]:
def get_readable_size(size_in_bytes):
    # Units to iterate through
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_in_bytes < 1024.0:
            return f"{size_in_bytes:.2f} {unit}"
        size_in_bytes /= 1024.0

In [3]:
def column_stats(df,column_name, topN=10, label_length=150):
    # normalize=True gives fractions (0.125), * 100 makes it percentages (12.5)
    counts = df[column_name].value_counts()
    percentages = df[column_name].value_counts(normalize=True) * 100

    unique_values = len(counts)
    if counts.empty:
        max_val_len = 0
        min_val_len = 0
        avg_val_len = 0
    else:
        max_val_len = counts.index.astype(str).str.len().max()
        min_val_len = counts.index.astype(str).str.len().min()
        avg_val_len = int(counts.index.astype(str).to_series().str.len().mean())

    print(f"\nColumn [{column_name}] has {unique_values:,} unique values, max/min content lenghts {max_val_len:,}/{min_val_len} ; avg lenght {avg_val_len:,}\n")
  # Define column widths once to keep them consistent
    if max_val_len > label_length:
        w1 = label_length
    else:
        w1 = max(max_val_len, 20) # Ensure it's at least 20 wide

   
    w2, w3 = 20, 15


    if unique_values > topN:
        print(f'Printing top {topN} records out of {unique_values:,}')

    # Print the Header
    divider = '-' * (w1 + w2 + w3 + 6)
    print(divider) 
    print(f"{'Value':<{w1}} | {'Total Records':<{w2}} | {'Percentage':<{w3}}")
    print(divider) 
    
    # Print the Data
    for i, ((label, count), percent) in enumerate(zip(counts.items(), percentages)):
        if i> topN:
            break

        # 1. Wrap the label into a list of lines; width=w1 tells it where to break the text
        wrapped_label = textwrap.wrap(str(label), width=w1)
        
        # 2. Print the first line with the data (count and percent)
        first_line = wrapped_label[0] if wrapped_label else ""
        print(f"{first_line:<{w1}} | {count:<{w2},} | {percent:.2f}%")
        
        # 3. Print the remaining lines of the label (if any)
        # We leave the Count and Percentage columns empty for these lines
        for extra_line in wrapped_label[1:]:
            print(f"{extra_line:<{w1}} | {'':<{w2}} | {'':<{w3}}")
        
        if len(wrapped_label) > 1:
            print(f"{'-'*w1:<{w1}} | {'-'*w2:<{w2}} | {'-'*w3:<{w3}}")


In [4]:
def get_records (df, query_field, query_value, rec_toPrint=5):
    
    records = df[df[query_field] == query_value ]

    if records.empty:
        print(f"No records found when {query_field}  == {query_value}")
        return None

    records_found = len(records)
    print(f"Records found {records_found:,} when {query_field}  == {query_value}\n")

    if records_found > rec_toPrint:
        print(f'Printing first {rec_toPrint} records out of {records_found:,}')

    for i, (index, record)  in enumerate(records.iterrows()):
        if i+1 > rec_toPrint:
            break

        for field, value in record.items():
            print(f"[Record={i+1}] | [Key={index}] | {field} : {value}")
        print('')
    # return records



In [5]:
def get_dataframe_stats_per_column(df):
    columns_list = df.columns.tolist()

    for column in columns_list:
        column_stats(df,column)

In [6]:
def df_stats(df):
    print(f"Dataset contains {df.shape[0]:,} rows and {df.shape[1]} columns.")

In [7]:
def dataset_missing_values(df):
    # 1. Calculate absolute missing values
    tot_missing = df.isnull().sum()

    # 2. Calculate percentage (missing / total rows * 100)
    percent_missing = (df.isnull().sum() / len(df)) * 100

    # 3. Combine into a clean DataFrame
    missing_report = pd.concat([tot_missing, percent_missing], axis=1)

    # 4. Rename columns for clarity
    missing_report.columns = ['tot_missing', '% of overall']

    # 5. Sort by most missing to least missing
    missing_report = missing_report.sort_values(by='tot_missing', ascending=False)

    print("Missing Values Report:\n")
    print(missing_report)